# Grader resolution check -- can the grader see real progression at 128 px / 256 px / full resolution?

A quick check before committing to 256 px for round 8 (notebook 09's pretraining at 256 px is ~20 h of Colab). **No
Generator, no training** -- it only grades REAL Tianjin baseline + 2-year follow-up photos with Module 1's graders.

Notebook 04_05 already measured how well the grader grades single photos at each size (round 5, EyePACS grader, kappa vs
Tianjin's clinical grades: full 0.586, 256 px 0.506, 128 px 0.463). Module 2 needs something different: does the grader see
the **change** between a baseline and its follow-up? For every eye-pair this notebook asks whether the grader rates the
follow-up higher than the baseline ("grade went up"):

- on eyes that **progressed** (clinical grade went up) -- should be high
- on eyes that stayed **stable** -- false alarms, should be low

**Separation** = "went up" on progressed minus "went up" on stable. That is what Module 2's evaluation can detect at best.
Run at 128 px and 256 px (the photos shrunk exactly as Module 2 sees them) and at full resolution (the grader's own
preprocessing of the original photo). All Tianjin patients are used -- no Generator, so nothing has to be held out.

**Reads** (nothing is written outside `module2_runs/grader_resolution_check_r8/`): round 5's Tianjin Module 1 cache and
laterality file, the EyePACS + FGADR graders (notebook 11). **Runtime:** ~20-30 min on an L4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER, REPO_NAME = 'Mieka068', 'DRProgression'
REPO_BRANCH = 'R7-GradeControl-Eval'
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'
DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f'Not found: {DRIVE_DATA_DIR}'

# Same folders as notebooks 06-10 in rounds 6/7: round 5's Module 1 cache and laterality file.
MODULE1_SOURCE = 'drgnet_combined'
RUN_SUFFIX = '_prefinals'
BASE_RUN_TAG = '_drgnet_comb' + RUN_SUFFIX
INPUT_RUN_TAG = BASE_RUN_TAG + '_r5'
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + INPUT_RUN_TAG)
TIANJIN_CACHE = os.path.join(MODULE1_CACHE_DIR, 'module1_outputs_tianjin.pt')
LATERALITY_DRIVE_PATH = os.path.join(MODULE1_CACHE_DIR, 'laterality_resolved_v2.csv')
for _p in (TIANJIN_CACHE, LATERALITY_DRIVE_PATH):
    assert os.path.isfile(_p), f'Not found: {_p} -- round 5 notebook 04_05 / 06 outputs'
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'drgnet_replication')
if os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'classification')):
    MODULE1_RUNS += RUN_SUFFIX
OUT_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs', 'grader_resolution_check_r8')
os.makedirs(OUT_DIR, exist_ok=True)
os.environ['DRP_RETINA_CROP'] = '1'   # the same retina crop as Module 2 (rounds 5-7)
print('Module 1 cache:', MODULE1_CACHE_DIR, '| graders:', MODULE1_RUNS, '| results:', OUT_DIR)

In [ ]:
# Tianjin only (FIRE / LongDR aren't needed).
import glob
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw
_m = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _m, 'No corrected_manifest.csv found -- see notebook 05'
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if os.path.dirname(_m[0]) != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(os.path.dirname(_m[0]), TIANJIN_DIR)
import shutil
shutil.copy(LATERALITY_DRIVE_PATH, os.path.join(TIANJIN_DIR, 'laterality_resolved.csv'))
print('Tianjin:', TIANJIN_DIR, '| laterality file copied')

In [ ]:
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
else:
    !cd /content/{REPO_NAME} && git pull origin {REPO_BRANCH}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch torchmetrics torch-fidelity opencv-python-headless

In [ ]:
# The two graders, chosen as in notebook 06: EyePACS (best_validation_weights.pt) and FGADR (final_weights.pt).
def _first(run, names):
    d = os.path.join(MODULE1_RUNS, 'classification', run, 'saves')
    return next((os.path.join(d, n) for n in names if os.path.isfile(os.path.join(d, n))), None)
EYEPACS_CKPT = _first('eyepacs', ['best_validation_weights.pt', 'final_weights.pt'])
FGADR_CKPT = _first('fgadr', ['final_weights.pt', 'best_validation_weights.pt'])
assert EYEPACS_CKPT, f'No EyePACS grader in {MODULE1_RUNS} -- notebook 11'
print('EyePACS grader:', EYEPACS_CKPT)
print('FGADR grader  :', FGADR_CKPT or 'not found -- only the EyePACS grader is checked')
SECOND_ARGS = f' --second-classifier-checkpoint "{FGADR_CKPT}" --second-classifier-norm fgadr' if FGADR_CKPT else ''

In [ ]:
# Three runs (each reused on a re-run unless its settings changed): 128 px, 256 px, full resolution.
import json as _json
RUNS = [('128px', 128, False), ('256px', 256, False), ('full', 128, True)]
%cd {REPO_CODE_DIR}
for _tag, _size, _native in RUNS:
    _out = os.path.join(OUT_DIR, f'real_followup_{_tag}.json')
    if os.path.isfile(_out):
        _old = _json.load(open(_out))
        if (_old.get('include_stable') and _old.get('image_size') == _size
                and bool(_old.get('real_followup_native')) == _native and bool(_old.get('graders', {}).get('second')) == bool(FGADR_CKPT)):
            print('✓ Reusing', _out)
            continue
    _native_arg = ' --real-followup-native' if _native else ''
    print(f'\n=== {_tag} ===')
    !python evaluate_trajectory.py --image-size {_size} --real-followup --include-stable{_native_arg} \
        --tianjin-dir "{TIANJIN_DIR}" --tianjin-module1-cache "{TIANJIN_CACHE}" \
        --classifier-checkpoint "{EYEPACS_CKPT}" --classifier-norm eyepacs{SECOND_ARGS} \
        --out "{_out}"

In [ ]:
# Table + decision. "went up" = the grader rates the follow-up higher than the baseline. Brackets: 95 % intervals.
import pandas as pd
GRADERS = [('primary', 'EyePACS')] + ([('second', 'FGADR')] if FGADR_CKPT else [])
res = {t: _json.load(open(os.path.join(OUT_DIR, f'real_followup_{t}.json'))) for t, _, _ in RUNS
       if os.path.isfile(os.path.join(OUT_DIR, f'real_followup_{t}.json'))}
def _e(r, key, g):
    return ((r.get('final_step') or {}).get(key) or {}).get(g) or {}
def _fmt(e, k):
    return None if not e.get('n') else f"{e[k + '_rate']:.0%} [{e[k + '_ci95'][0]:.0%}-{e[k + '_ci95'][1]:.0%}]"
rows, sep = [], {}
for t, _, _ in RUNS:
    if t not in res:
        continue
    for g, name in GRADERS:
        prog, stab = _e(res[t], 'all', g), _e(res[t], 'step_0', g)
        s = (prog['went_up_rate'] - stab['went_up_rate']) if prog.get('n') and stab.get('n') else None
        sep[(t, g)] = s
        rows.append({'resolution': t, 'grader': name, 'progressed n': prog.get('n'), 'stable n': stab.get('n'),
                     'went up | progressed': _fmt(prog, 'went_up'), 'went up | stable (false alarm)': _fmt(stab, 'went_up'),
                     'separation': None if s is None else f'{s:+.0%}',
                     'exact grade | progressed': _fmt(prog, 'agree'), 'exact grade | stable': _fmt(stab, 'agree')})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, 'grader_resolution_check.csv'), index=False)
display(df)

s128, s256, sfull = (sep.get((t, 'primary')) for t in ('128px', '256px', 'full'))
print('\nDecision (EyePACS separation: went up on progressed minus went up on stable):')
if None in (s128, s256, sfull):
    print('  incomplete -- a run is missing (read the outputs above)')
else:
    print(f'  128 px {s128:+.0%} | 256 px {s256:+.0%} | full resolution {sfull:+.0%}')
    if s256 - s128 >= 0.10:
        print('  -> 256 px clearly helps: worth notebook 09\'s 256 px pretraining for round 8.')
    elif sfull - s128 >= 0.10:
        print('  -> Detail matters, but 256 px recovers little of it. 256 px alone is unlikely to rescue grade-based '
              'progression; weigh the ~20 h pretraining against that.')
    else:
        print('  -> Resolution is not the bottleneck: even at full resolution the grader barely separates progressed from '
              'stable eyes. Don\'t spend the 256 px pretraining on grade-based progression; report the grader limit.')
if FGADR_CKPT and sep.get(('full', 'second')) is not None:
    print(f"  FGADR separation at full resolution: {sep[('full', 'second')]:+.0%} "
          '(round 5\'s notebook 04_05 found its grades barely relate to Tianjin\'s -- check its exact-grade column)')

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')